In [2]:
import duckdb
import sys
from pathlib import Path
import pandas as pd 

ROOT_DIR = Path.cwd().parent

if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))
    
    
from src.db.sql_runner import run_query



# Antes de começar a Inteligência de Negócio, validar o dataset e realizar limpeza de acordo com as regras de negócio

In [4]:
df_nulos = run_query("01_nulls_check.sql")
df_nulos

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,column_type,pct_nulos,valores_unicos
0,DeviceInfo,VARCHAR,79.91,1665
1,id_13,DOUBLE,78.44,60
2,id_16,VARCHAR,78.10,2
3,V218,DOUBLE,77.91,402
4,V223,DOUBLE,77.91,18
...,...,...,...,...
357,V297,DOUBLE,0.00,15
358,V298,DOUBLE,0.00,123
359,V299,DOUBLE,0.00,54
360,categoria_provedor_comprador,VARCHAR,0.00,7


In [4]:
colunas_descarte = df_nulos[df_nulos['pct_nulos'] > 85]['column_name'].tolist()

colunas_mantidas = df_nulos[df_nulos['pct_nulos'] <= 85]['column_name'].tolist()

print(f'Total de {len(df_nulos)} colunas originais')
print(f'Total de {len(colunas_descarte)} colunas para descarte (>85% nulos)')
print(f'Total de {len(colunas_mantidas)} colunas mantidas (<=85% nulos)')

NameError: name 'df_nulos' is not defined

In [5]:
df_emails = run_query("02_ordenar_emails.sql")
df_emails

,P_emaildomain,qtde_emails
0,gmail.com,228355
1,yahoo.com,100934
2,NaN,94456
3,hotmail.com,45250
4,anonymous.com,36998
5,aol.com,28289
6,comcast.net,7888
7,icloud.com,6267
8,outlook.com,5096
9,msn.com,4092


In [6]:
agrupar_emails_comprador = run_query('03_agrupar_emails_comprador.sql')
agrupar_emails_comprador

,qtde_por_provedor,categoria_provedor
0,228851,google
1,36998,anonymous
2,59477,microsoft
3,105969,yahoo
4,56564,outro
5,94456,missing
6,8225,apple


In [7]:
agrupar_emails_destino = run_query('04_agrupar_emails_destino.sql')
agrupar_emails_destino

,qtde_por_provedor,categoria_provedor
0,57242,google
1,20529,anonymous
2,453249,missing
3,9777,outro
4,2172,apple
5,33604,microsoft
6,13967,yahoo


In [8]:
colunas_descarte = df_nulos[df_nulos['pct_nulos'] > 85]['column_name'].tolist()
colunas_descarte_str = ', '.join(colunas_descarte)
colunas_descarte_str

''

In [9]:
view_limpa = run_query("05_view_limpa.sql", colunas_descarte_str=colunas_descarte_str)
view_limpa

ParserException: Parser Error: syntax error at or near ")"

LINE 3:     EXCLUDE (),
                     ^

In [10]:
query_limpa = run_query('06_test_view.sql')
query_limpa

CatalogException: Catalog Error: Table with name vw_train_clean does not exist!
Did you mean "pg_tables"?

LINE 1: SELECT * FROM vw_train_clean
                      ^

In [11]:
duckdb.execute(r"COPY vw_train_clean TO 'caminho para o arquivo' (FORMAT PARQUET)")

CatalogException: Catalog Error: Table with name vw_train_clean does not exist!
Did you mean "pg_tables"?

# Documentação: Tratamento de Dados e Engenharia de Features (Camada Silver)
Objetivo: Registrar o pipeline de limpeza, governança de nulos e normalização de categorias aplicados no dataset de detecção de fraudes para a consolidação da camada Silver (train_clean.parquet).
# 1. Regras de Governança e Qualidade de Dados (DQ)
Regra: Identificação e descarte de colunas que apresentem taxa de valores ausentes (nulos) superior a 85%.
Motivação: Variáveis com densidade de dados inferior a 15% introduzem ruído ao modelo e reduzem a eficiência computacional, sem agregar sinal preditivo relevante.
Resultado: 74 colunas excederam o limite do SLA e foram removidas dinamicamente do dataset através da cláusula EXCLUDE do DuckDB.
# 2. Engenharia de Features e Categorização (ID)
### Normalização de Domínios de E-mail (ID-02)
As colunas originais de e-mail do comprador (P_emaildomain) e do destinatário (R_emaildomain) apresentavam alta cardinalidade e fragmentação (ex: variações como gmail.com, gmail, hotmail.com, hotmail.co.uk).
Regra Aplicada: Mapeamento via sintaxe condicional CASE WHEN agrupando os domínios em 7 categorias estratégicas:

google

microsoft

yahoo

apple

anonymous

missing (preservação explícita de registros nulos para análise de risco)

outro (provedores corporativos/raros)
# Novas Features Geradas:
categoria_provedor_comprador: Provedor tratado do e-mail de quem realiza a compra.

categoria_provedor_destino: Provedor tratado do e-mail do destinatário (sinal valioso para fraudes em gift cards e entregas a terceiros).
# 4. Pipeline de Materialização e Exportação
Construção da View Virtual: As regras de negócio foram unificadas em uma VIEW lógica no DuckDB (vw_train_clean), evitando duplicação de dados na memória RAM durante os testes.

Exportação de Alta Performance: O arquivo final foi persistido diretamente do motor C++ do DuckDB para o disco, eliminando gargalos do PyArrow e otimizando o tempo de gravação.
Caminho de Destino: data/processed/train_clean.parquet
Tempo de Execução: 10,8 segundos para gravação de 590.540 linhas x 362 colunas.

# Pulando de camada (Bronze para Silver)
PARQUET_PATH do sql_runner alterado para parquet_limpo

In [12]:
risco_provedor_comprador = run_query('07_calculo_fraude_email_comp.sql')
risco_provedor_comprador

,categoria_provedor_comprador,total_transacoes_email_comprador,total_fraudes_email_comprador,porcentagem_fraude_email_comprador
0,microsoft,59477,3170.0,5.33
1,google,228851,9954.0,4.35
2,missing,94456,2790.0,2.95
3,apple,8225,238.0,2.89
4,anonymous,36998,859.0,2.32
5,outro,56564,1280.0,2.26
6,yahoo,105969,2372.0,2.24


In [13]:
risco_provedor_destino = run_query('08_calculo_fraude_email_dest.sql')
risco_provedor_destino

,categoria_provedor_destino,total_transacoes_email_destino,total_fraudes_email_destino,porcentagem_fraude_email_destino
0,google,57242,6811.0,11.90
1,apple,2172,193.0,8.89
2,microsoft,33604,2714.0,8.08
3,yahoo,13967,644.0,4.61
4,anonymous,20529,598.0,2.91
5,outro,9777,267.0,2.73
6,missing,453249,9436.0,2.08


In [17]:
matriz_cruzada = run_query('09_matriz_cruz_email.sql')
matriz_cruzada

,categoria_provedor_comprador,categoria_provedor_destino,total_emails,total_fraudes,taxa_fraude_pct
0,microsoft,apple,62,14.0,22.58
1,google,google,44526,6141.0,13.79
2,apple,apple,1026,97.0,9.45
3,yahoo,apple,131,12.0,9.16
4,missing,apple,286,26.0,9.09
5,microsoft,microsoft,30771,2632.0,8.55
6,missing,google,4959,419.0,8.45
7,google,apple,428,34.0,7.94
8,outro,apple,138,8.0,5.80
9,apple,google,328,18.0,5.49


In [20]:
comparacao_fraude_legitima = run_query('10_comp_fraude_legitima.sql')
comparacao_fraude_legitima

,isFraud,total_transacoes,ticket_medio,mediana,desvio_padrao,variancia,valor_minimo,valor_maximo
0,0,569877,134.51,68.5,239.40,57310.00,0.251,31937.391
1,1,20663,149.24,75.0,232.21,53922.49,0.292,5191.000
